# 1.a - AquaCrop pilot: rainfed maize on a few cells, against the SVM suitability

**Purpose.** Set up and check the AquaCrop chain before running it on the grid: input files written from ERA5, one
AquaCrop project per cell with one run per season, results read back, and a first comparison with the climate
suitability of the SVM (4.g). Model: FAO **AquaCrop 7.3** stand-alone (KU Leuven build, release v7.3_typofix), crop
file `MaizeGDD.CRO` of the FAO database (crop cycle in growing degree-days, so that the cycle length follows the
temperature of each cell). Helpers in `2_AQUACROP/aquacrop_utils.py`. Writes only in `outputs/aquacrop/pilot_maize/`
and in a work folder (`WORK`).

**Inputs per cell** (SVM grid, EC-Earth3 0.7 deg):
- climate: daily ERA5 tasmax, tasmin, pr, averaged over the 0.25 deg pixels whose centre is in the cell, weighted by
  their land share (pixels of the HWSD raster with a soil unit): the sea pixels of coastal cells would give a maritime
  climate; ET0 of Hargreaves (as 1.1.d / 11.a, no radiation); CO2 of the FAO Mauna Loa file (observed concentration);
- sowing: GGCMI phase 3 planting day of the cell (nearest grid point with a value where the cell has none);
- cultivar: `MaizeGDD.CRO` is a maize of Davis (California) needing 1700 degree-days to maturity. As gridded crop
  models do (GGCMI), the cultivar is adapted to the local season: the degree-days from the GGCMI planting to the GGCMI
  maturity day (mean over the seasons, AquaCrop method 3) replace the 1700, every degree-day duration of the crop file
  but emergence scaled by the same factor and the canopy coefficients per degree-day divided by it
  (`aquacrop_utils.write_crop_variant`); the physiology (water productivity, harvest index, stress responses) is that
  of the FAO file. A cultivar is a property of the crop, not of the climate: where maize is not grown, the nearest GGCMI
  season applied to a colder climate would give cultivars that do not exist (140-230 degree-days in Lapland), so the
  cultivar is not earlier than the earliest one grown today (5th percentile of the local degree-days of the cells where
  maize is grown);
- soil: the layers of HWSD v2 (D1-D7, down to 2 m or to the first layer without data in shallow soils) of the dominant
  component of the dominant soil mapping unit of the cell, grouped into 5 horizons (0-20, 20-40, 40-80, 80-150,
  150-200 cm); saturation, field capacity, wilting point and saturated conductivity of the fine earth from sand,
  clay, organic matter and bulk density with the pedotransfer functions of Saxton & Rawls (2006), as regional AquaCrop
  set-ups use HWSD with pedotransfer functions (de Roos et al. 2021, GMD 14, 7309); gravel from the HWSD coarse
  fragments (AquaCrop reduces the water storage itself); curve number, readily evaporable water and capillary-rise
  parameters with the rules of the FAO database (checked on its 12 default soils); where HWSD has the bedrock below
  the last layer (shallow soils), a horizon of penetrability 0 % down to 2 m, the restrictive layer of AquaCrop (the
  thickness of the horizons alone does not limit the roots: tested);
- no irrigation, no field management (no fertility stress), no groundwater.

**Seasons.** One continuous water balance per cell (`KeepSWC`: each run starts with the soil water left by the
previous one); run k from 30 days before sowing to 31 days before the next sowing; the first season starts at field
capacity and is dropped (spin-up). The crop cycle ends at maturity (growing degree-days of the crop file) as computed
by AquaCrop.

**AquaCrop as it is.** The results of AquaCrop are kept as they are, including where the crop cannot reach maturity
(AquaCrop then prolongs the cycle to the end of the run and still reports a yield; tested: 9.3 t/ha with 1450
degree-days a year for 1700 required): the comparison is between the two models as they work. With the cultivar of
the local season this concerns mainly the cells where maize is not grown. The degree-days of
each season (AquaCrop method 3, from sowing to the end of the run) are reported for information only. Where no
degree-day is accumulated at all (ice), the programme does not end: every cell runs alone with a time limit, and a cell
that does not finish has no result (`no result`).

**Cells.** `N_PER_GROUP` cells where maize is grown today (presence of the training route) and as many where it is not,
spread over the SVM probability on ERA5 (cells at evenly spaced ranks), inside the area of the daily ERA5 file
available (the local file covers Europe only).

**Outputs.** Per cell: mean rainfed dry yield over the seasons, its variability, cycle length, water and temperature
stress; yield against the SVM probability, today's cultivated against the others.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
import sys, glob, sqlite3, tempfile, time
from datetime import date
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
sys.path.insert(0, str(BASE)); sys.path.insert(0, str(BASE / "2_AQUACROP"))
import crop_utils as cu
import aquacrop_utils as aq
CFG = cu.load_config()

CROP = "Maize_rf"
CROP_FILE = "MaizeGDD.CRO"                       # FAO database, crop cycle in growing degree-days
GGCMI = "mai"                                    # GGCMI phase 3 calendar
CELL_AREA_THR = 0.0005                           # presence route of the SVM (cellareafrac 0.05 %)
SVM_SET = "A_60d_no_rad"                         # 4.g predictions (scenario ERA5)
YEARS = (1981, 2014)                             # harvest years; the first season is the spin-up
N_PER_GROUP = 10                                 # cells grown today / not grown
SOW_LEAD = 30                                    # days of simulation before sowing
WORK = Path(os.environ.get("AQUACROP_WORK", Path(tempfile.gettempdir()) / "aquacrop_pilot_maize"))
N_WORKERS = int(os.environ.get("AQUACROP_WORKERS", "4"))   # cells run in parallel
TIMEOUT = 120                                   # s per cell (only to stop a cell where AquaCrop does not end)

ROUTE = cu.route(CELL_AREA_THR)
OUT = cu.out_dir(CFG, "aquacrop", "pilot_maize")
GRID = cu.svm_grid(CFG)
print(f"AquaCrop {aq.AQ_VERSION}: {aq.EXE} | crop {CROP_FILE} | work {WORK} | written in {OUT}")

## 1. Cells: SVM probability on ERA5, today's maize, inside the daily ERA5 area

In [ ]:
with xr.open_dataset(cu.out_dir(CFG, "4_svm_predict", ROUTE, SVM_SET) / f"suitability_{CROP}.nc") as ds:
    P = ds["suit_proba"].sel(scenario="ERA5").values
cm = cu.crop_mask(CFG, CROP)
PRES = (cm["presence"].values == 1) & (cu.cell_area_frac(CFG, cm["area_ha"].values) > CELL_AREA_THR)
LAND = cu.land_mask(CFG).values

E5 = {v: xr.open_dataset(cu.data_file(CFG, "era5", v)) for v in ("tasmax", "tasmin", "pr")}
e5lat, e5lon = E5["tasmax"]["lat"].values, E5["tasmax"]["lon"].values
inside = ((GRID["lat_edges"][:-1] >= e5lat.min())[:, None] & (GRID["lat_edges"][1:] <= e5lat.max())[:, None]
          & (GRID["lon_edges"][:-1] >= e5lon.min())[None, :] & (GRID["lon_edges"][1:] <= e5lon.max())[None, :])
cand = LAND & inside & np.isfinite(P)
rows = []
for grp, m in (("grown", cand & PRES), ("not grown", cand & ~PRES)):
    i, j = np.nonzero(m)
    order = np.argsort(P[i, j])
    pick = order[np.unique(np.round(np.linspace(0, order.size - 1, N_PER_GROUP)).astype(int))]
    rows += [{"cell": f"{grp[0]}{k:02d}", "group": grp, "i": int(i[p]), "j": int(j[p]), "lat": float(GRID["lat"][i[p]]),
              "lon": float(GRID["lon"][j[p]]), "svm_p": float(P[i[p], j[p]])} for k, p in enumerate(pick)]
CELLS = pd.DataFrame(rows).set_index("cell")
print(f"{cand.sum()} candidate cells ({(cand & PRES).sum()} grown) inside lat {e5lat.min()}..{e5lat.max()}, lon {e5lon.min()}..{e5lon.max()}")
display(CELLS)

## 2. Inputs per cell: daily climate, sowing day, soil

In [ ]:
def cell_box(i, j):
    return (GRID["lat_edges"][i], GRID["lat_edges"][i + 1], GRID["lon_edges"][j], GRID["lon_edges"][j + 1])


import rasterio
from rasterio.windows import from_bounds
HW_RASTER = cu.data_file(CFG, "hwsd_raster")
E5_STEP = float(abs(e5lat[1] - e5lat[0]))


def land_share(lat, lon):
    # share of the HWSD 30" pixels with a soil unit (not sea) in the ERA5 pixel centred on (lat, lon)
    h = E5_STEP / 2
    with rasterio.open(HW_RASTER) as r:
        a = r.read(1, window=from_bounds(lon - h, lat - h, lon + h, lat + h, r.transform))
    return float(np.mean((a != 65535) & (a > 0))) if a.size else 0.0


def cell_climate(i, j):
    # daily ERA5 averaged over the 0.25 deg pixels whose centre is in the cell, weighted by their land share
    la0, la1, lo0, lo1 = cell_box(i, j)
    out, w = {}, None
    for v, ds in E5.items():
        da = ds[v]
        sl = da.sel(lat=slice(la0, la1) if da.lat[0] < da.lat[-1] else slice(la1, la0), lon=slice(lo0, lo1))
        if w is None:
            w = xr.DataArray([[land_share(a, b) for b in sl.lon.values] for a in sl.lat.values], dims=("lat", "lon"),
                             coords={"lat": sl.lat, "lon": sl.lon})
            if float(w.sum()) == 0:
                w = xr.ones_like(w)
        out[v] = sl.weighted(w).mean(("lat", "lon")).load()
    t = pd.DatetimeIndex(out["tasmax"]["time"].values).normalize()
    return t, {v: out[v].values.astype(float) for v in out}, float(w.mean())


# sowing day: GGCMI phase 3 planting day (nearest point with a value)
with xr.open_dataset(glob.glob(str(Path(cu.data_files(CFG, "ggcmi_calendar")[0]).parent / f"{GGCMI}_rf_ggcmi_crop_calendar_phase3*.nc4"))[0],
                     decode_times=False) as ds:
    G = ds["planting_day"].values; GM = ds["maturity_day"].values; glat, glon = ds["lat"].values, ds["lon"].values
gi, gj = np.nonzero(np.isfinite(G) & np.isfinite(GM))


def sowing_doy(lat, lon):
    # GGCMI planting and maturity day of the nearest point with both, distance (deg)
    k = np.argmin((glat[gi] - lat) ** 2 + ((glon[gj] - lon + 180) % 360 - 180) ** 2 * np.cos(np.radians(lat)) ** 2)
    return (int(round(G[gi[k], gj[k]])), int(round(GM[gi[k], gj[k]])),
            float(np.hypot(glat[gi[k]] - lat, glon[gj[k]] - lon)))


# soil: HWSD v2 layers of the dominant component of the dominant soil mapping unit, Saxton & Rawls horizons
import rasterio
from rasterio.windows import from_bounds
HW_RASTER = cu.data_file(CFG, "hwsd_raster")
HW_DB = Path(cu.data_file(CFG, "hwsd_mdb")).with_suffix(".sqlite")
con = cu.hwsd_connect(HW_DB if HW_DB.exists() else cu.data_file(CFG, "hwsd_mdb"))
LAY = pd.read_sql("SELECT HWSD2_SMU_ID, SEQUENCE, SHARE, LAYER, TOPDEP, BOTDEP, SAND, SILT, CLAY, ORG_CARBON, BULK, COARSE "
                  "FROM HWSD2_LAYERS", con)
for k in ("SAND", "SILT", "CLAY", "ORG_CARBON", "BULK", "COARSE", "TOPDEP", "BOTDEP"):
    LAY[k] = pd.to_numeric(LAY[k], errors="coerce")
# negative values kept: -7 = below the bedrock (read by aquacrop_utils.hwsd_profile), -3 / -9 = no data
D1OK = LAY[(LAY["LAYER"] == "D1") & (LAY["SAND"] >= 0)]              # components with a topsoil description
DOMC = (D1OK.sort_values(["HWSD2_SMU_ID", "SHARE", "SEQUENCE"], ascending=[True, False, True])
        .drop_duplicates("HWSD2_SMU_ID")[["HWSD2_SMU_ID", "SEQUENCE"]])
LAYG = LAY.groupby(["HWSD2_SMU_ID", "SEQUENCE"])
DOM_LAYERS = {smu: LAYG.get_group((smu, seq)) for smu, seq in DOMC.itertuples(index=False)}


def soil_profile(i, j):
    # dominant soil mapping unit with layer data in the cell -> AquaCrop horizons (None: water, ice, rock)
    la0, la1, lo0, lo1 = cell_box(i, j)
    with rasterio.open(HW_RASTER) as r:
        a = r.read(1, window=from_bounds(lo0, la0, lo1, la1, r.transform))
    for smu in pd.Series(a[(a != 65535) & (a > 0)]).value_counts().index:
        if int(smu) in DOM_LAYERS:
            hz = aq.hwsd_profile(DOM_LAYERS[int(smu)])
            if hz:
                return int(smu), hz
    return None, None


GP = aq.crop_gdd_parameters(CROP_FILE)
PREP, INFO = {}, []
t0 = time.time()
for cell, c in CELLS.iterrows():
    t, X, land = cell_climate(c.i, c.j)
    keep = (t.year >= YEARS[0] - 1) & (t.year <= YEARS[1] + 1)
    t = t[keep]; X = {v: x[keep] for v, x in X.items()}
    eto = aq.eto_hargreaves(X["tasmax"], X["tasmin"], c.lat, t.dayofyear.values)
    sow, mat, dist = sowing_doy(c.lat, c.lon)
    smu, hz = soil_profile(c.i, c.j)
    # one run per season: from SOW_LEAD days before sowing to the day before the next run (continuous water balance)
    sowd = [aq.daynr(date(y, 1, 1)) + sow - 1 for y in range(YEARS[0], YEARS[1] + 1)]
    runs = [{"sim1": s - SOW_LEAD, "simN": (sowd[k + 1] - SOW_LEAD - 1) if k + 1 < len(sowd) else s + 365 - SOW_LEAD - 1,
             "crop1": s, "cropN": s + 1, "keep": True} for k, s in enumerate(sowd)]
    d0, dN = aq.daynr(t[0].date()), aq.daynr(t[-1].date())
    runs = [r for r in runs if r["sim1"] >= d0 and r["simN"] <= dN]
    gdd = aq.daily_gdd(X["tasmin"], X["tasmax"], GP["tbase"], GP["tupper"])
    # cultivar of the local season: degree-days from the GGCMI planting to the GGCMI maturity day, mean of the seasons
    glen = (mat - sow) % 365 or 365
    gdd_local = float(np.mean([gdd[r["crop1"] - d0:r["crop1"] - d0 + glen].sum() for r in runs]))
    for r in runs:                                   # degree-days from sowing to the end of the run, and of the run (information)
        r["gdd_season"] = float(gdd[r["crop1"] - d0:r["simN"] - d0 + 1].sum())
        r["gdd_run"] = float(gdd[r["sim1"] - d0:r["simN"] - d0 + 1].sum())
    PREP[cell] = {"first": t[0].date(), "X": X, "eto": eto, "runs": runs, "hz": hz, "title": f"{cell} ({c.lat:.2f}, {c.lon:.2f})",
                  "gdd_local": gdd_local}
    taw = None if hz is None else sum((h["fc"] - h["wp"]) * h["thickness"] * 10 * (1 - h["gravel"] / 100) for h in hz)
    INFO.append({"cell": cell, "status": "no soil" if hz is None else "run", "sowing_doy": sow, "maturity_doy": mat,
                 "ggcmi_distance_deg": round(dist, 2), "land_share": land, "gdd_cultivar": gdd_local,
                 "cultivar_factor": gdd_local / GP["gdd_maturity"],
                 "rock_at_m": (sum(h["thickness"] for h in hz) if hz and hz[-1].get("rock_below") else None) if hz else None,
                 "smu": smu, "soil_depth_m": None if hz is None else sum(h["thickness"] for h in hz),
                 "topsoil": None if hz is None else hz[0]["texture"], "taw_mm": taw,
                 "rain_mm_yr": X["pr"].sum() / (t.size / 365.25), "tmean": float(np.mean((X["tasmax"] + X["tasmin"]) / 2)),
                 "eto_mm_yr": eto.sum() / (t.size / 365.25), "seasons": len(runs),
                 "gdd_season_mean": float(np.mean([r["gdd_season"] for r in runs])),
                 "seasons_below_cultivar_gdd_pct": 100 * np.mean([r["gdd_season"] < gdd_local for r in runs])})
CELLS = CELLS.join(pd.DataFrame(INFO).set_index("cell"))
# a cultivar is a property of the crop: below the earliest cultivar grown today (5th percentile of the local degree-days
# of the cells where maize is grown) the local GGCMI season (taken from the nearest cropped point) is not a cultivar
CULTIVAR_MIN = float(np.percentile(CELLS.loc[CELLS["group"] == "grown", "gdd_cultivar"], 5))
for cell in PREP:
    PREP[cell]["gdd_cultivar"] = int(round(max(PREP[cell]["gdd_local"], CULTIVAR_MIN)))
CELLS["gdd_cultivar_used"] = [PREP[c]["gdd_cultivar"] for c in CELLS.index]
print(f"earliest cultivar: {CULTIVAR_MIN:.0f} degree-days (5th percentile of the grown cells)")
print(f"inputs prepared for {len(PREP)} cells ({time.time() - t0:.0f} s); crop {CROP_FILE}: {GP}")
display(CELLS.round(2))


def write_inputs(work, cell):
    p = PREP[cell]
    files = aq.write_climate(work / "DATA", cell, p["first"], p["X"]["tasmin"], p["X"]["tasmax"], p["eto"], p["X"]["pr"],
                             title=p["title"])
    aq.write_soil(work / "DATA" / f"{cell}.SOL", p["title"], p["hz"])
    aq.write_crop_variant(CROP_FILE, work / "DATA" / f"{cell}.CRO", p["gdd_cultivar"],
                          title=f"{CROP_FILE[:-4]}, cultivar of {p['gdd_cultivar']} degree-days to maturity ({cell})")
    aq.write_project(work / "LIST" / f"{cell}.PRM", f"{cell} maize pilot", p["runs"], files, f"{cell}.CRO", f"{cell}.SOL")
    return f"{cell}.PRM"

## 3. AquaCrop runs and seasonal results

In [ ]:
t0 = time.time()
torun = CELLS.index[CELLS["status"] == "run"].tolist()
RAW, TIMED_OUT = aq.run_cells(WORK, torun, write_inputs, [CROP_FILE], n_workers=N_WORKERS, timeout=TIMEOUT)
print(f"AquaCrop: {len(torun)} cells in {time.time() - t0:.0f} s ({N_WORKERS} workers); did not end within {TIMEOUT} s: {TIMED_OUT}")
CELLS.loc[TIMED_OUT, "status"] = "no result"
rows = []
for cell in CELLS.index:
    df = RAW.get(cell)
    for k, r in enumerate(PREP[cell]["runs"]):
        row = {"cell": cell, "season": k + 1, "sowing": aq.date_of(r["crop1"]), "gdd_season": r["gdd_season"], "gdd_run": r["gdd_run"]}
        if df is not None and k < len(df):
            row.update(df.iloc[k][["Rain", "ETo", "GD", "Tr", "Cycle", "TempStr", "StoStr", "BioMass", "HI", "Y(dry)"]].to_dict())
        rows.append(row)
S = pd.DataFrame(rows)
S["harvest_year"] = [d.year for d in S["sowing"]]
S["yield_t_ha"] = S["Y(dry)"] if "Y(dry)" in S else np.nan                 # AquaCrop's dry yield, as it is
chk = S.dropna(subset=["GD"])                                              # check: our degree-days of the run vs AquaCrop's
print("degree-days of the runs, ours - AquaCrop: median %.1f, max |.| %.1f" % ((chk["gdd_run"] - chk["GD"]).median(),
                                                                                  (chk["gdd_run"] - chk["GD"]).abs().max()))
S = S[S["season"] > 1]                                                     # spin-up season dropped
S.to_csv(OUT / "seasons_pilot_maize.csv", index=False)
g = S.groupby("cell")
R = pd.DataFrame({"yield_t_ha": g["yield_t_ha"].mean(), "yield_cv": g["yield_t_ha"].std() / g["yield_t_ha"].mean().replace(0, np.nan),
                  "failed_seasons_pct": 100 * g["yield_t_ha"].apply(lambda y: (y < 0.5).mean() if y.notna().any() else np.nan),
                  "cycle_days": g["Cycle"].mean(), "stomata_stress_pct": g["StoStr"].mean(),
                  "temperature_stress_pct": g["TempStr"].mean(), "transpiration_mm": g["Tr"].mean()})
RES = CELLS.join(R)
RES.to_csv(OUT / "cells_pilot_maize.csv")
with pd.option_context("display.width", 250, "display.max_columns", 40):
    display(RES.round(2))

## 4. Rainfed yield against the SVM probability

In [ ]:
RES = RES.dropna(subset=["yield_t_ha"])                    # cells without an AquaCrop result left out
rho, prho = stats.spearmanr(RES["svm_p"], RES["yield_t_ha"])
g = RES["group"] == "grown"
auc_y = stats.mannwhitneyu(RES.loc[g, "yield_t_ha"], RES.loc[~g, "yield_t_ha"]).statistic / (g.sum() * (~g).sum())
auc_p = stats.mannwhitneyu(RES.loc[g, "svm_p"], RES.loc[~g, "svm_p"]).statistic / (g.sum() * (~g).sum())
print(f"Spearman(SVM probability, AquaCrop yield) = {rho:.2f} (p = {prho:.3f}), {len(RES)} cells")
print(f"grown vs not grown today: AUC of the AquaCrop yield {auc_y:.2f}, of the SVM probability {auc_p:.2f} (pilot cells only)")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for grp, col in (("grown", "C2"), ("not grown", "C7")):
    s = RES[RES["group"] == grp]
    axes[0].scatter(s["svm_p"], s["yield_t_ha"], c=col, label=f"{grp} today", s=40)
    for cell, r in s.iterrows():
        axes[0].annotate(cell, (r["svm_p"], r["yield_t_ha"]), fontsize=7)
axes[0].set_xlabel("SVM probability (ERA5)"); axes[0].set_ylabel("AquaCrop rainfed dry yield (t/ha, mean)"); axes[0].legend(fontsize=8)
sc = axes[1].scatter(RES["lon"], RES["lat"], c=RES["yield_t_ha"], cmap="YlGn", s=60, edgecolors=np.where(g, "k", "0.6"))
plt.colorbar(sc, ax=axes[1], label="yield (t/ha)"); axes[1].set_title("cells (black edge: grown today)", fontsize=9)
for cell in RES.index:
    y = S.loc[S["cell"] == cell].sort_values("harvest_year")
    axes[2].plot(y["harvest_year"], y["yield_t_ha"], color="C2" if RES.loc[cell, "group"] == "grown" else "C7", lw=0.8, alpha=0.7)
axes[2].set_xlabel("harvest year"); axes[2].set_ylabel("yield (t/ha)"); axes[2].set_title("seasons per cell", fontsize=9)
plt.tight_layout(); plt.savefig(OUT / "pilot_maize_yield_vs_svm.png", dpi=110); plt.show()